# ⚽ Notebook 09 — React Frontend Dashboard

Builds the ScoutIQ React app and writes it to `src/frontend/`.

**Pages:**
- 🏠 Home — search + quick stats
- 🎯 Scout — filter by position/style/budget
- 🔄 Similar — find players who play like X
- 💎 Hidden Gems — undervalued talent
- ⚖️ Compare — side-by-side stats

In [ ]:
from pathlib import Path
import subprocess, sys, os

ROOT     = Path('C:/Users/Lenovo/Scouti_Koora')
FRONTEND = ROOT / 'src' / 'frontend'
FRONTEND.mkdir(parents=True, exist_ok=True)

print(f'Frontend dir: {FRONTEND}')
print('Writing React app...')

In [ ]:
# ── index.html ──────────────────────────────────────────────────────
html = '''<!DOCTYPE html>\n",
    "<html lang=\"en\">\n",
    "<head>\n",
    "  <meta charset=\"UTF-8\" />\n",
    "  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\" />\n",
    "  <title>ScoutIQ — Football Talent Discovery</title>\n",
    "  <link rel=\"preconnect\" href=\"https://fonts.googleapis.com\">\n",
    "  <link href=\"https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=Space+Grotesk:wght@400;500;700&display=swap\" rel=\"stylesheet\">\n",
    "  <script src=\"https://cdnjs.cloudflare.com/ajax/libs/react/18.2.0/umd/react.production.min.js\"></script>\n",
    "  <script src=\"https://cdnjs.cloudflare.com/ajax/libs/react-dom/18.2.0/umd/react-dom.production.min.js\"></script>\n",
    "  <script src=\"https://cdnjs.cloudflare.com/ajax/libs/recharts/2.8.0/Recharts.min.js\"></script>\n",
    "  <script src=\"https://cdnjs.cloudflare.com/ajax/libs/babel-standalone/7.23.2/babel.min.js\"></script>\n",
    "  <style>\n",
    "    *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }\n",
    "    :root {\n",
    "      --bg:       #0a0e1a;\n",
    "      --surface:  #111827;\n",
    "      --surface2: #1a2235;\n",
    "      --border:   #1e2d45;\n",
    "      --accent:   #00d4ff;\n",
    "      --accent2:  #7c3aed;\n",
    "      --gold:     #f59e0b;\n",
    "      --green:    #10b981;\n",
    "      --red:      #ef4444;\n",
    "      --text:     #f0f4ff;\n",
    "      --muted:    #6b7fa3;\n",
    "      --font-display: \"Space Grotesk\", sans-serif;\n",
    "      --font-body:    \"Inter\", sans-serif;\n",
    "    }\n",
    "    html { scroll-behavior: smooth; }\n",
    "    body { background: var(--bg); color: var(--text); font-family: var(--font-body); min-height: 100vh; }\n",
    "\n",
    "    /* ── Layout ── */\n",
    "    .app { display: flex; flex-direction: column; min-height: 100vh; }\n",
    "    .nav {\n",
    "      display: flex; align-items: center; gap: 2rem;\n",
    "      padding: 0 2rem; height: 60px;\n",
    "      background: rgba(10,14,26,0.95);\n",
    "      border-bottom: 1px solid var(--border);\n",
    "      position: sticky; top: 0; z-index: 100;\n",
    "      backdrop-filter: blur(12px);\n",
    "    }\n",
    "    .nav-logo {\n",
    "      font-family: var(--font-display);\n",
    "      font-size: 1.2rem; font-weight: 700;\n",
    "      color: var(--accent); letter-spacing: -0.02em;\n",
    "      cursor: pointer; white-space: nowrap;\n",
    "    }\n",
    "    .nav-links { display: flex; gap: 0.25rem; }\n",
    "    .nav-btn {\n",
    "      background: none; border: none; cursor: pointer;\n",
    "      color: var(--muted); font-family: var(--font-body);\n",
    "      font-size: 0.85rem; font-weight: 500;\n",
    "      padding: 0.4rem 0.8rem; border-radius: 6px;\n",
    "      transition: color 0.15s, background 0.15s;\n",
    "    }\n",
    "    .nav-btn:hover { color: var(--text); background: var(--surface); }\n",
    "    .nav-btn.active { color: var(--accent); background: rgba(0,212,255,0.08); }\n",
    "    .main { flex: 1; padding: 2rem; max-width: 1280px; margin: 0 auto; width: 100%; }\n",
    "\n",
    "    /* ── Cards ── */\n",
    "    .card {\n",
    "      background: var(--surface); border: 1px solid var(--border);\n",
    "      border-radius: 12px; padding: 1.5rem;\n",
    "    }\n",
    "    .card-sm { padding: 1rem; border-radius: 10px; }\n",
    "\n",
    "    /* ── Player Card ── */\n",
    "    .player-card {\n",
    "      background: var(--surface); border: 1px solid var(--border);\n",
    "      border-radius: 12px; padding: 1.25rem;\n",
    "      transition: border-color 0.2s, transform 0.15s;\n",
    "      cursor: pointer;\n",
    "    }\n",
    "    .player-card:hover { border-color: var(--accent); transform: translateY(-2px); }\n",
    "    .player-name { font-family: var(--font-display); font-weight: 700; font-size: 1rem; margin-bottom: 0.25rem; }\n",
    "    .player-meta { font-size: 0.78rem; color: var(--muted); display: flex; gap: 0.5rem; flex-wrap: wrap; }\n",
    "    .player-stats { display: grid; grid-template-columns: repeat(3,1fr); gap: 0.5rem; margin-top: 1rem; }\n",
    "    .stat-box { background: var(--surface2); border-radius: 8px; padding: 0.5rem; text-align: center; }\n",
    "    .stat-val { font-family: var(--font-display); font-size: 1.1rem; font-weight: 700; color: var(--accent); }\n",
    "    .stat-lbl { font-size: 0.65rem; color: var(--muted); text-transform: uppercase; letter-spacing: 0.05em; margin-top: 1px; }\n",
    "\n",
    "    /* ── Tags ── */\n",
    "    .tag {\n",
    "      display: inline-block; font-size: 0.7rem; font-weight: 600;\n",
    "      padding: 0.2rem 0.55rem; border-radius: 4px;\n",
    "      background: rgba(0,212,255,0.12); color: var(--accent);\n",
    "      letter-spacing: 0.03em;\n",
    "    }\n",
    "    .tag-gold  { background: rgba(245,158,11,0.12); color: var(--gold); }\n",
    "    .tag-green { background: rgba(16,185,129,0.12); color: var(--green); }\n",
    "    .tag-purple{ background: rgba(124,58,237,0.12); color: #a78bfa; }\n",
    "\n",
    "    /* ── Inputs ── */\n",
    "    input, select {\n",
    "      background: var(--surface2); border: 1px solid var(--border);\n",
    "      color: var(--text); font-family: var(--font-body); font-size: 0.88rem;\n",
    "      padding: 0.6rem 0.9rem; border-radius: 8px; outline: none;\n",
    "      transition: border-color 0.15s;\n",
    "    }\n",
    "    input:focus, select:focus { border-color: var(--accent); }\n",
    "    input::placeholder { color: var(--muted); }\n",
    "    select option { background: var(--surface); }\n",
    "    label { font-size: 0.78rem; color: var(--muted); font-weight: 500; display: block; margin-bottom: 0.35rem; text-transform: uppercase; letter-spacing: 0.05em; }\n",
    "\n",
    "    /* ── Buttons ── */\n",
    "    .btn {\n",
    "      background: var(--accent); color: #000; font-weight: 700;\n",
    "      font-family: var(--font-display); font-size: 0.85rem;\n",
    "      border: none; border-radius: 8px; padding: 0.65rem 1.4rem;\n",
    "      cursor: pointer; transition: opacity 0.15s, transform 0.1s;\n",
    "      letter-spacing: 0.01em;\n",
    "    }\n",
    "    .btn:hover { opacity: 0.88; }\n",
    "    .btn:active { transform: scale(0.97); }\n",
    "    .btn-ghost {\n",
    "      background: none; border: 1px solid var(--border);\n",
    "      color: var(--text);\n",
    "    }\n",
    "    .btn-ghost:hover { border-color: var(--accent); color: var(--accent); background: rgba(0,212,255,0.06); }\n",
    "\n",
    "    /* ── Grid ── */\n",
    "    .grid-2 { display: grid; grid-template-columns: repeat(2,1fr); gap: 1rem; }\n",
    "    .grid-3 { display: grid; grid-template-columns: repeat(3,1fr); gap: 1rem; }\n",
    "    .grid-4 { display: grid; grid-template-columns: repeat(4,1fr); gap: 1rem; }\n",
    "    @media (max-width: 900px) {\n",
    "      .grid-4, .grid-3 { grid-template-columns: repeat(2,1fr); }\n",
    "      .grid-2 { grid-template-columns: 1fr; }\n",
    "    }\n",
    "    @media (max-width: 560px) {\n",
    "      .grid-4, .grid-3, .grid-2 { grid-template-columns: 1fr; }\n",
    "    }\n",
    "\n",
    "    /* ── Misc ── */\n",
    "    .section-title {\n",
    "      font-family: var(--font-display); font-size: 1.5rem;\n",
    "      font-weight: 700; letter-spacing: -0.03em; margin-bottom: 0.25rem;\n",
    "    }\n",
    "    .section-sub { color: var(--muted); font-size: 0.88rem; margin-bottom: 1.5rem; }\n",
    "    .divider { border: none; border-top: 1px solid var(--border); margin: 2rem 0; }\n",
    "    .flex { display: flex; }\n",
    "    .items-center { align-items: center; }\n",
    "    .gap-1 { gap: 0.5rem; } .gap-2 { gap: 1rem; } .gap-3 { gap: 1.5rem; }\n",
    "    .mb-1 { margin-bottom: 0.5rem; } .mb-2 { margin-bottom: 1rem; } .mb-3 { margin-bottom: 1.5rem; }\n",
    "    .mt-2 { margin-top: 1rem; }\n",
    "    .w-full { width: 100%; }\n",
    "    .text-accent { color: var(--accent); }\n",
    "    .text-muted  { color: var(--muted);  }\n",
    "    .text-gold   { color: var(--gold);   }\n",
    "    .text-green  { color: var(--green);  }\n",
    "    .empty-state { text-align: center; padding: 3rem 1rem; color: var(--muted); }\n",
    "    .empty-state .icon { font-size: 2.5rem; margin-bottom: 0.75rem; }\n",
    "    .talent-bar { height: 4px; border-radius: 2px; background: var(--border); margin-top: 0.5rem; }\n",
    "    .talent-fill { height: 100%; border-radius: 2px; background: linear-gradient(90deg, var(--accent2), var(--accent)); transition: width 0.6s ease; }\n",
    "    .spinner { display: inline-block; width: 20px; height: 20px; border: 2px solid var(--border); border-top-color: var(--accent); border-radius: 50%; animation: spin 0.7s linear infinite; }\n",
    "    @keyframes spin { to { transform: rotate(360deg); } }\n",
    "    .filter-row { display: flex; flex-wrap: wrap; gap: 0.75rem; align-items: flex-end; }\n",
    "    .filter-field { display: flex; flex-direction: column; min-width: 140px; }\n",
    "    .hero {\n",
    "      text-align: center; padding: 3rem 1rem 2rem;\n",
    "      background: radial-gradient(ellipse 60% 40% at 50% 0%, rgba(0,212,255,0.07) 0%, transparent 70%);\n",
    "    }\n",
    "    .hero-title {\n",
    "      font-family: var(--font-display); font-size: clamp(2rem, 5vw, 3.5rem);\n",
    "      font-weight: 700; letter-spacing: -0.04em; line-height: 1.1;\n",
    "      margin-bottom: 0.75rem;\n",
    "    }\n",
    "    .hero-title span { color: var(--accent); }\n",
    "    .hero-sub { color: var(--muted); font-size: 1rem; max-width: 480px; margin: 0 auto 2rem; line-height: 1.6; }\n",
    "    .search-wrap { position: relative; max-width: 500px; margin: 0 auto; }\n",
    "    .search-input {\n",
    "      width: 100%; padding: 0.85rem 3rem 0.85rem 1.2rem;\n",
    "      font-size: 1rem; border-radius: 12px;\n",
    "      border: 1px solid var(--border);\n",
    "      background: var(--surface);\n",
    "    }\n",
    "    .search-icon { position: absolute; right: 1rem; top: 50%; transform: translateY(-50%); color: var(--muted); pointer-events: none; font-size: 1rem; }\n",
    "    .search-results {\n",
    "      position: absolute; top: calc(100% + 6px); left: 0; right: 0;\n",
    "      background: var(--surface); border: 1px solid var(--border);\n",
    "      border-radius: 10px; z-index: 200; overflow: hidden;\n",
    "      box-shadow: 0 8px 32px rgba(0,0,0,0.4);\n",
    "    }\n",
    "    .search-item {\n",
    "      padding: 0.75rem 1rem; cursor: pointer;\n",
    "      display: flex; justify-content: space-between; align-items: center;\n",
    "      border-bottom: 1px solid var(--border); font-size: 0.88rem;\n",
    "      transition: background 0.1s;\n",
    "    }\n",
    "    .search-item:last-child { border-bottom: none; }\n",
    "    .search-item:hover { background: var(--surface2); }\n",
    "    .kpi-row { display: grid; grid-template-columns: repeat(4,1fr); gap: 1rem; margin-bottom: 2rem; }\n",
    "    @media(max-width:700px){ .kpi-row { grid-template-columns: repeat(2,1fr); } }\n",
    "    .kpi {\n",
    "      background: var(--surface); border: 1px solid var(--border);\n",
    "      border-radius: 12px; padding: 1.25rem;\n",
    "      display: flex; flex-direction: column; gap: 0.25rem;\n",
    "    }\n",
    "    .kpi-val { font-family: var(--font-display); font-size: 2rem; font-weight: 700; line-height: 1; }\n",
    "    .kpi-lbl { font-size: 0.75rem; color: var(--muted); text-transform: uppercase; letter-spacing: 0.05em; }\n",
    "    .compare-table { width: 100%; border-collapse: collapse; font-size: 0.88rem; }\n",
    "    .compare-table th, .compare-table td { padding: 0.6rem 1rem; text-align: right; border-bottom: 1px solid var(--border); }\n",
    "    .compare-table th:first-child, .compare-table td:first-child { text-align: left; color: var(--muted); }\n",
    "    .compare-table th { color: var(--accent); font-family: var(--font-display); font-weight: 600; }\n",
    "    .compare-table tr:last-child td { border-bottom: none; }\n",
    "    .better { color: var(--green); font-weight: 600; }\n",
    "    .similarity-bar { display: flex; align-items: center; gap: 0.75rem; }\n",
    "    .sim-fill { height: 6px; border-radius: 3px; background: linear-gradient(90deg, var(--accent2), var(--accent)); }\n",
    "    .sim-track { flex: 1; height: 6px; border-radius: 3px; background: var(--border); overflow: hidden; }\n",
    "    .pagination { display: flex; justify-content: center; gap: 0.5rem; margin-top: 2rem; }\n",
    "    .page-btn { background: var(--surface); border: 1px solid var(--border); color: var(--text); padding: 0.4rem 0.8rem; border-radius: 6px; cursor: pointer; font-size: 0.82rem; }\n",
    "    .page-btn.active { border-color: var(--accent); color: var(--accent); }\n",
    "    .page-btn:hover:not(.active) { border-color: var(--muted); }\n",
    "  </style>\n",
    "</head>\n",
    "<body>\n",
    "  <div id=\"root\"></div>\n",
    "  <script type=\"text/babel\">\n",
    "    const { useState, useEffect, useCallback, useRef } = React;\n",
    "    const API = \"http://localhost:8000\";\n",
    "\n",
    "    // ── Helpers ──────────────────────────────────────────────────────\n",
    "    const fmt = (v, decimals=2) => v == null ? \"—\" : Number(v).toFixed(decimals);\n",
    "    const fmtM = v => v == null ? \"—\" : v >= 1e6 ? `€${(v/1e6).toFixed(1)}M` : v >= 1e3 ? `€${(v/1e3).toFixed(0)}K` : `€${v}`;\n",
    "    const talentColor = s => s >= 85 ? \"#f59e0b\" : s >= 70 ? \"#00d4ff\" : s >= 55 ? \"#10b981\" : \"#6b7fa3\";\n",
    "    const talentLabel = s => s >= 85 ? \"Elite\" : s >= 70 ? \"World Class\" : s >= 55 ? \"Top Player\" : s >= 40 ? \"Good\" : \"Average\";\n",
    "    const posColor = p => ({Forward:\"#ef4444\",Midfielder:\"#f59e0b\",Defender:\"#3b82f6\",Goalkeeper:\"#10b981\"}[p] || \"#6b7fa3\");\n",
    "\n",
    "    // ── Fetch helper ─────────────────────────────────────────────────\n",
    "    const apiFetch = async (path) => {\n",
    "      const r = await fetch(API + path);\n",
    "      if (!r.ok) throw new Error(`${r.status}`);\n",
    "      return r.json();\n",
    "    };\n",
    "\n",
    "    // ── StatBox ──────────────────────────────────────────────────────\n",
    "    const StatBox = ({val, lbl}) => (\n",
    "      <div className=\"stat-box\">\n",
    "        <div className=\"stat-val\">{val}</div>\n",
    "        <div className=\"stat-lbl\">{lbl}</div>\n",
    "      </div>\n",
    "    );\n",
    "\n",
    "    // ── PlayerCard ───────────────────────────────────────────────────\n",
    "    const PlayerCard = ({ p, onClick }) => {\n",
    "      const score = p.predicted_talent ?? p.overall_score;\n",
    "      return (\n",
    "        <div className=\"player-card\" onClick={() => onClick && onClick(p)}>\n",
    "          <div className=\"flex items-center gap-1 mb-1\">\n",
    "            <span className=\"tag\" style={{background:`${posColor(p.primary_pos)}22`,color:posColor(p.primary_pos)}}>\n",
    "              {p.primary_pos?.[0] ?? \"?\"}\n",
    "            </span>\n",
    "            {p.style_label && <span className=\"tag tag-purple\">{p.style_label}</span>}\n",
    "          </div>\n",
    "          <div className=\"player-name\">{p.Player}</div>\n",
    "          <div className=\"player-meta\">\n",
    "            <span>{p.Squad}</span>\n",
    "            <span>·</span>\n",
    "            <span>{p.league}</span>\n",
    "            <span>·</span>\n",
    "            <span>Age {p.Age}</span>\n",
    "          </div>\n",
    "          {score != null && (\n",
    "            <div className=\"mt-2\">\n",
    "              <div className=\"flex items-center gap-1\" style={{justifyContent:\"space-between\",marginBottom:4}}>\n",
    "                <span style={{fontSize:\"0.72rem\",color:\"var(--muted)\"}}>TALENT SCORE</span>\n",
    "                <span style={{fontSize:\"0.85rem\",fontWeight:700,color:talentColor(score)}}>{fmt(score,1)}</span>\n",
    "              </div>\n",
    "              <div className=\"talent-bar\">\n",
    "                <div className=\"talent-fill\" style={{width:`${Math.min(score,100)}%`}}></div>\n",
    "              </div>\n",
    "            </div>\n",
    "          )}\n",
    "          <div className=\"player-stats\">\n",
    "            <StatBox val={fmt(p.goals_p90)} lbl=\"Gls/90\" />\n",
    "            <StatBox val={fmt(p.assists_p90)} lbl=\"Ast/90\" />\n",
    "            <StatBox val={fmtM(p.market_value_eur)} lbl=\"Value\" />\n",
    "          </div>\n",
    "        </div>\n",
    "      );\n",
    "    };\n",
    "\n",
    "    // ── Spinner ──────────────────────────────────────────────────────\n",
    "    const Spinner = () => <div style={{textAlign:\"center\",padding:\"2rem\"}}><div className=\"spinner\"></div></div>;\n",
    "\n",
    "    // ── Search bar ───────────────────────────────────────────────────\n",
    "    const SearchBar = ({ onSelect, placeholder = \"Search player...\" }) => {\n",
    "      const [q, setQ] = useState(\"\");\n",
    "      const [results, setResults] = useState([]);\n",
    "      const [open, setOpen] = useState(false);\n",
    "      const timer = useRef(null);\n",
    "      useEffect(() => {\n",
    "        if (q.length < 2) { setResults([]); setOpen(false); return; }\n",
    "        clearTimeout(timer.current);\n",
    "        timer.current = setTimeout(async () => {\n",
    "          const data = await apiFetch(`/players?q=${encodeURIComponent(q)}&top_n=6`).catch(()=>[]);\n",
    "          setResults(data); setOpen(true);\n",
    "        }, 300);\n",
    "      }, [q]);\n",
    "      return (\n",
    "        <div className=\"search-wrap\">\n",
    "          <input className=\"search-input\" value={q} onChange={e=>setQ(e.target.value)}\n",
    "            placeholder={placeholder} onBlur={()=>setTimeout(()=>setOpen(false),200)} />\n",
    "          <span className=\"search-icon\">⚽</span>\n",
    "          {open && results.length > 0 && (\n",
    "            <div className=\"search-results\">\n",
    "              {results.map((r,i) => (\n",
    "                <div key={i} className=\"search-item\" onMouseDown={()=>{onSelect(r);setQ(r.Player);setOpen(false);}}>\n",
    "                  <div>\n",
    "                    <strong>{r.Player}</strong>\n",
    "                    <span style={{color:\"var(--muted)\",marginLeft:8,fontSize:\"0.78rem\"}}>{r.Squad} · {r.league}</span>\n",
    "                  </div>\n",
    "                  <span style={{color:posColor(r.primary_pos),fontSize:\"0.75rem\",fontWeight:600}}>{r.primary_pos}</span>\n",
    "                </div>\n",
    "              ))}\n",
    "            </div>\n",
    "          )}\n",
    "        </div>\n",
    "      );\n",
    "    };\n",
    "\n",
    "    // ── HOME PAGE ────────────────────────────────────────────────────\n",
    "    const HomePage = ({ setPage }) => {\n",
    "      const [health, setHealth] = useState(null);\n",
    "      useEffect(() => { apiFetch(\"/health\").then(setHealth).catch(()=>{}); }, []);\n",
    "      return (\n",
    "        <div>\n",
    "          <div className=\"hero\">\n",
    "            <div className=\"hero-title\">Discover football<br/><span>talent before anyone else</span></div>\n",
    "            <div className=\"hero-sub\">AI-powered scouting across 5 top leagues, 5 seasons, 4,817 players ranked by style, talent, and value.</div>\n",
    "            <SearchBar onSelect={()=>{}} placeholder=\"Search any player...\" />\n",
    "          </div>\n",
    "\n",
    "          {health && (\n",
    "            <div className=\"kpi-row mt-2\">\n",
    "              <div className=\"kpi\"><div className=\"kpi-val text-accent\">{health.players.toLocaleString()}</div><div className=\"kpi-lbl\">Players indexed</div></div>\n",
    "              <div className=\"kpi\"><div className=\"kpi-val text-gold\">5</div><div className=\"kpi-lbl\">Top leagues</div></div>\n",
    "              <div className=\"kpi\"><div className=\"kpi-val\" style={{color:\"var(--green)\"}}>{health.style_labels.length}</div><div className=\"kpi-lbl\">Play styles</div></div>\n",
    "              <div className=\"kpi\"><div className=\"kpi-val\" style={{color:\"#a78bfa\"}}>3</div><div className=\"kpi-lbl\">AI models</div></div>\n",
    "            </div>\n",
    "          )}\n",
    "\n",
    "          <div className=\"grid-2 gap-2\">\n",
    "            {[{icon:\"🎯\",title:\"Find by style\",desc:\"Filter by position, play style, budget and age to surface hidden talent.\",page:\"recommend\"},\n",
    "              {icon:\"🔄\",title:\"Similar players\",desc:\"Enter any player — get the 10 who play most like them using KNN cosine similarity.\",page:\"similar\"},\n",
    "              {icon:\"💎\",title:\"Hidden gems\",desc:\"High XGBoost talent score, low market value. Undervalued players scouts miss.\",page:\"gems\"},\n",
    "              {icon:\"⚖️\",title:\"Compare\",desc:\"Side-by-side stats and radar chart comparison between any two players.\",page:\"compare\"},\n",
    "            ].map(({icon,title,desc,page}) => (\n",
    "              <div key={page} className=\"card\" style={{cursor:\"pointer\",transition:\"border-color 0.2s\"}} onClick={()=>setPage(page)}\n",
    "                onMouseEnter={e=>e.currentTarget.style.borderColor=\"var(--accent)\"}\n",
    "                onMouseLeave={e=>e.currentTarget.style.borderColor=\"var(--border)\"}>\n",
    "                <div style={{fontSize:\"1.8rem\",marginBottom:\"0.5rem\"}}>{icon}</div>\n",
    "                <div style={{fontFamily:\"var(--font-display)\",fontWeight:700,fontSize:\"1rem\",marginBottom:\"0.35rem\"}}>{title}</div>\n",
    "                <div style={{color:\"var(--muted)\",fontSize:\"0.85rem\",lineHeight:1.5}}>{desc}</div>\n",
    "              </div>\n",
    "            ))}\n",
    "          </div>\n",
    "\n",
    "          {health && (\n",
    "            <div className=\"card mt-2\">\n",
    "              <div style={{fontFamily:\"var(--font-display)\",fontWeight:600,marginBottom:\"0.75rem\",fontSize:\"0.85rem\",color:\"var(--muted)\",textTransform:\"uppercase\",letterSpacing:\"0.05em\"}}>Available play styles</div>\n",
    "              <div style={{display:\"flex\",flexWrap:\"wrap\",gap:\"0.4rem\"}}>\n",
    "                {health.style_labels.map(s=><span key={s} className=\"tag tag-purple\">{s}</span>)}\n",
    "              </div>\n",
    "            </div>\n",
    "          )}\n",
    "        </div>\n",
    "      );\n",
    "    };\n",
    "\n",
    "    // ── RECOMMEND PAGE ───────────────────────────────────────────────\n",
    "    const RecommendPage = () => {\n",
    "      const [filters, setFilters] = useState({position:\"\",style:\"\",max_age:\"\",max_budget:\"\",leagues:\"\"});\n",
    "      const [results, setResults] = useState([]);\n",
    "      const [loading, setLoading] = useState(false);\n",
    "      const [styles, setStyles] = useState([]);\n",
    "      useEffect(() => { apiFetch(\"/styles\").then(setStyles).catch(()=>{}); }, []);\n",
    "      const search = async () => {\n",
    "        setLoading(true);\n",
    "        const p = new URLSearchParams();\n",
    "        if (filters.position) p.set(\"position\", filters.position);\n",
    "        if (filters.style)    p.set(\"style\",    filters.style);\n",
    "        if (filters.max_age)  p.set(\"max_age\",  filters.max_age);\n",
    "        if (filters.max_budget) p.set(\"max_budget_eur\", filters.max_budget * 1e6);\n",
    "        if (filters.leagues)  p.set(\"leagues\",  filters.leagues);\n",
    "        p.set(\"top_n\", 12);\n",
    "        const data = await apiFetch(`/recommend?${p}`).catch(()=>[]);\n",
    "        setResults(data); setLoading(false);\n",
    "      };\n",
    "      const set = (k,v) => setFilters(f=>({...f,[k]:v}));\n",
    "      return (\n",
    "        <div>\n",
    "          <div className=\"section-title\">🎯 Scout by profile</div>\n",
    "          <div className=\"section-sub\">Filter across 4,817 players to find your next signing</div>\n",
    "          <div className=\"card mb-3\">\n",
    "            <div className=\"filter-row\">\n",
    "              <div className=\"filter-field\">\n",
    "                <label>Position</label>\n",
    "                <select value={filters.position} onChange={e=>set(\"position\",e.target.value)}>\n",
    "                  <option value=\"\">All</option>\n",
    "                  {[\"forward\",\"midfielder\",\"defender\",\"goalkeeper\"].map(p=><option key={p} value={p}>{p[0].toUpperCase()+p.slice(1)}</option>)}\n",
    "                </select>\n",
    "              </div>\n",
    "              <div className=\"filter-field\">\n",
    "                <label>Style</label>\n",
    "                <select value={filters.style} onChange={e=>set(\"style\",e.target.value)}>\n",
    "                  <option value=\"\">All styles</option>\n",
    "                  {styles.map(s=><option key={s} value={s}>{s}</option>)}\n",
    "                </select>\n",
    "              </div>\n",
    "              <div className=\"filter-field\">\n",
    "                <label>Max age</label>\n",
    "                <input type=\"number\" min=\"16\" max=\"40\" placeholder=\"e.g. 25\" value={filters.max_age} onChange={e=>set(\"max_age\",e.target.value)} />\n",
    "              </div>\n",
    "              <div className=\"filter-field\">\n",
    "                <label>Max budget (€M)</label>\n",
    "                <input type=\"number\" min=\"0\" placeholder=\"e.g. 30\" value={filters.max_budget} onChange={e=>set(\"max_budget\",e.target.value)} />\n",
    "              </div>\n",
    "              <div className=\"filter-field\">\n",
    "                <label>League</label>\n",
    "                <select value={filters.leagues} onChange={e=>set(\"leagues\",e.target.value)}>\n",
    "                  <option value=\"\">All leagues</option>\n",
    "                  {[\"EPL\",\"LaLiga\",\"SerieA\",\"Bundesliga\",\"Ligue1\"].map(l=><option key={l} value={l}>{l}</option>)}\n",
    "                </select>\n",
    "              </div>\n",
    "              <button className=\"btn\" onClick={search}>Search</button>\n",
    "            </div>\n",
    "          </div>\n",
    "          {loading ? <Spinner /> : results.length === 0 ? (\n",
    "            <div className=\"empty-state\"><div className=\"icon\">🔍</div><div>Set filters and press Search</div></div>\n",
    "          ) : (\n",
    "            <>\n",
    "              <div className=\"text-muted mb-2\" style={{fontSize:\"0.82rem\"}}>{results.length} players found</div>\n",
    "              <div className=\"grid-3\">{results.map((p,i)=><PlayerCard key={i} p={p} />)}</div>\n",
    "            </>\n",
    "          )}\n",
    "        </div>\n",
    "      );\n",
    "    };\n",
    "\n",
    "    // ── SIMILAR PAGE ─────────────────────────────────────────────────\n",
    "    const SimilarPage = () => {\n",
    "      const [selected, setSelected] = useState(null);\n",
    "      const [results,  setResults]  = useState([]);\n",
    "      const [loading,  setLoading]  = useState(false);\n",
    "      const search = async (player) => {\n",
    "        if (!player) return;\n",
    "        setSelected(player); setLoading(true);\n",
    "        const data = await apiFetch(`/similar/${encodeURIComponent(player.Player)}?top_n=9`).catch(()=>[]);\n",
    "        setResults(Array.isArray(data) ? data : []); setLoading(false);\n",
    "      };\n",
    "      return (\n",
    "        <div>\n",
    "          <div className=\"section-title\">🔄 Find similar players</div>\n",
    "          <div className=\"section-sub\">Enter a player — KNN cosine similarity finds who plays like them</div>\n",
    "          <div className=\"card mb-3\">\n",
    "            <SearchBar onSelect={search} placeholder=\"Enter reference player...\" />\n",
    "          </div>\n",
    "          {selected && <div className=\"mb-2\" style={{fontSize:\"0.88rem\",color:\"var(--muted)\"}}>Players similar to <strong style={{color:\"var(--text)\"}}>{selected.Player}</strong></div>}\n",
    "          {loading ? <Spinner /> : results.length === 0 ? (\n",
    "            <div className=\"empty-state\"><div className=\"icon\">🔄</div><div>Search a player to find similar profiles</div></div>\n",
    "          ) : (\n",
    "            <div className=\"grid-3\">\n",
    "              {results.map((r,i) => (\n",
    "                <div key={i} className=\"card card-sm\">\n",
    "                  <div className=\"flex items-center gap-1 mb-1\">\n",
    "                    <span className=\"tag tag-green\" style={{fontFamily:\"var(--font-display)\",fontSize:\"0.8rem\"}}>{r.similarity_pct}%</span>\n",
    "                    <span style={{fontSize:\"0.72rem\",color:\"var(--muted)\"}}>similarity</span>\n",
    "                  </div>\n",
    "                  <div className=\"player-name\">{r.Player}</div>\n",
    "                  <div className=\"player-meta\"><span>{r.Squad}</span><span>·</span><span>{r.league}</span></div>\n",
    "                  <div className=\"similarity-bar mt-2\">\n",
    "                    <div className=\"sim-track\"><div className=\"sim-fill\" style={{width:`${r.similarity_pct}%`}}></div></div>\n",
    "                  </div>\n",
    "                </div>\n",
    "              ))}\n",
    "            </div>\n",
    "          )}\n",
    "        </div>\n",
    "      );\n",
    "    };\n",
    "\n",
    "    // ── HIDDEN GEMS PAGE ─────────────────────────────────────────────\n",
    "    const GemsPage = () => {\n",
    "      const [filters, setFilters] = useState({position:\"\",max_age:\"25\",max_value:\"20\",min_talent:\"60\"});\n",
    "      const [results, setResults] = useState([]);\n",
    "      const [loading, setLoading] = useState(false);\n",
    "      const set = (k,v) => setFilters(f=>({...f,[k]:v}));\n",
    "      const search = async () => {\n",
    "        setLoading(true);\n",
    "        const p = new URLSearchParams();\n",
    "        if (filters.position)  p.set(\"position\", filters.position);\n",
    "        if (filters.max_age)   p.set(\"max_age\",  filters.max_age);\n",
    "        if (filters.max_value) p.set(\"max_value_eur\", filters.max_value * 1e6);\n",
    "        if (filters.min_talent)p.set(\"min_talent_score\", filters.min_talent);\n",
    "        p.set(\"top_n\", 12);\n",
    "        const data = await apiFetch(`/hidden-gems?${p}`).catch(()=>[]);\n",
    "        setResults(data); setLoading(false);\n",
    "      };\n",
    "      useEffect(() => { search(); }, []);\n",
    "      return (\n",
    "        <div>\n",
    "          <div className=\"section-title\">💎 Hidden gems</div>\n",
    "          <div className=\"section-sub\">High talent score, low market value — players scouts overlook</div>\n",
    "          <div className=\"card mb-3\">\n",
    "            <div className=\"filter-row\">\n",
    "              <div className=\"filter-field\">\n",
    "                <label>Position</label>\n",
    "                <select value={filters.position} onChange={e=>set(\"position\",e.target.value)}>\n",
    "                  <option value=\"\">All</option>\n",
    "                  {[\"forward\",\"midfielder\",\"defender\",\"goalkeeper\"].map(p=><option key={p} value={p}>{p[0].toUpperCase()+p.slice(1)}</option>)}\n",
    "                </select>\n",
    "              </div>\n",
    "              <div className=\"filter-field\">\n",
    "                <label>Max age</label>\n",
    "                <input type=\"number\" value={filters.max_age} onChange={e=>set(\"max_age\",e.target.value)} />\n",
    "              </div>\n",
    "              <div className=\"filter-field\">\n",
    "                <label>Max value (€M)</label>\n",
    "                <input type=\"number\" value={filters.max_value} onChange={e=>set(\"max_value\",e.target.value)} />\n",
    "              </div>\n",
    "              <div className=\"filter-field\">\n",
    "                <label>Min talent score</label>\n",
    "                <input type=\"number\" min=\"0\" max=\"100\" value={filters.min_talent} onChange={e=>set(\"min_talent\",e.target.value)} />\n",
    "              </div>\n",
    "              <button className=\"btn\" onClick={search}>Search</button>\n",
    "            </div>\n",
    "          </div>\n",
    "          {loading ? <Spinner /> : results.length === 0 ? (\n",
    "            <div className=\"empty-state\"><div className=\"icon\">💎</div><div>No gems found with these filters</div></div>\n",
    "          ) : (\n",
    "            <>\n",
    "              <div className=\"text-muted mb-2\" style={{fontSize:\"0.82rem\"}}>{results.length} gems found</div>\n",
    "              <div className=\"grid-3\">{results.map((p,i)=>{\n",
    "                const score = p.predicted_talent ?? p.overall_score;\n",
    "                return (\n",
    "                  <div key={i} className=\"player-card\">\n",
    "                    <div className=\"flex items-center gap-1 mb-1\">\n",
    "                      <span className=\"tag tag-gold\">💎 GEM</span>\n",
    "                      {p.style_label && <span className=\"tag tag-purple\">{p.style_label}</span>}\n",
    "                    </div>\n",
    "                    <div className=\"player-name\">{p.Player}</div>\n",
    "                    <div className=\"player-meta\"><span>{p.Squad}</span><span>·</span><span>{p.league}</span><span>·</span><span>Age {p.Age}</span></div>\n",
    "                    {score != null && (\n",
    "                      <div className=\"mt-2\">\n",
    "                        <div className=\"flex items-center gap-1\" style={{justifyContent:\"space-between\",marginBottom:4}}>\n",
    "                          <span style={{fontSize:\"0.72rem\",color:\"var(--muted)\"}}>TALENT</span>\n",
    "                          <span style={{fontSize:\"0.85rem\",fontWeight:700,color:talentColor(score)}}>{fmt(score,1)}</span>\n",
    "                        </div>\n",
    "                        <div className=\"talent-bar\"><div className=\"talent-fill\" style={{width:`${Math.min(score,100)}%`}}></div></div>\n",
    "                      </div>\n",
    "                    )}\n",
    "                    <div className=\"player-stats\">\n",
    "                      <StatBox val={fmt(p.goals_p90)} lbl=\"Gls/90\" />\n",
    "                      <StatBox val={fmt(p.assists_p90)} lbl=\"Ast/90\" />\n",
    "                      <StatBox val={fmtM(p.market_value_eur)} lbl=\"Value\" />\n",
    "                    </div>\n",
    "                  </div>\n",
    "                );\n",
    "              })}\n",
    "              </div>\n",
    "            </>\n",
    "          )}\n",
    "        </div>\n",
    "      );\n",
    "    };\n",
    "\n",
    "    // ── COMPARE PAGE ─────────────────────────────────────────────────\n",
    "    const ComparePage = () => {\n",
    "      const [playerA, setPlayerA] = useState(null);\n",
    "      const [playerB, setPlayerB] = useState(null);\n",
    "      const [result,  setResult]  = useState(null);\n",
    "      const [loading, setLoading] = useState(false);\n",
    "      const compare = async () => {\n",
    "        if (!playerA || !playerB) return;\n",
    "        setLoading(true);\n",
    "        const data = await apiFetch(`/compare?player_a=${encodeURIComponent(playerA.Player)}&player_b=${encodeURIComponent(playerB.Player)}`).catch(()=>null);\n",
    "        setResult(data); setLoading(false);\n",
    "      };\n",
    "      const METRICS = [\n",
    "        {key:\"goals_p90\",     label:\"Goals / 90\",    fmt:v=>fmt(v,2)},\n",
    "        {key:\"assists_p90\",   label:\"Assists / 90\",  fmt:v=>fmt(v,2)},\n",
    "        {key:\"g_a_p90\",       label:\"G+A / 90\",      fmt:v=>fmt(v,2)},\n",
    "        {key:\"shots_p90\",     label:\"Shots / 90\",    fmt:v=>fmt(v,2)},\n",
    "        {key:\"shot_accuracy_pct\",label:\"Shot acc %\", fmt:v=>v?`${fmt(v,1)}%`:\"—\"},\n",
    "        {key:\"tackles_won_p90\",label:\"Tackles / 90\",fmt:v=>fmt(v,2)},\n",
    "        {key:\"interceptions_p90\",label:\"Int / 90\",  fmt:v=>fmt(v,2)},\n",
    "        {key:\"availability_pct\",label:\"Availability\",fmt:v=>v?`${fmt(v,1)}%`:\"—\"},\n",
    "        {key:\"market_value_eur\",label:\"Market value\",fmt:fmtM},\n",
    "        {key:\"predicted_talent\",label:\"Talent score\",fmt:v=>fmt(v,1)},\n",
    "      ];\n",
    "      const getVal = (row, key) => row ? row[key] : null;\n",
    "      return (\n",
    "        <div>\n",
    "          <div className=\"section-title\">⚖️ Compare players</div>\n",
    "          <div className=\"section-sub\">Side-by-side stat comparison</div>\n",
    "          <div className=\"card mb-3\">\n",
    "            <div className=\"grid-2 gap-2 mb-2\">\n",
    "              <div>\n",
    "                <label>Player A</label>\n",
    "                <SearchBar onSelect={setPlayerA} placeholder=\"Search player A...\" />\n",
    "                {playerA && <div style={{marginTop:6,fontSize:\"0.82rem\",color:\"var(--accent)\"}}>{playerA.Player}</div>}\n",
    "              </div>\n",
    "              <div>\n",
    "                <label>Player B</label>\n",
    "                <SearchBar onSelect={setPlayerB} placeholder=\"Search player B...\" />\n",
    "                {playerB && <div style={{marginTop:6,fontSize:\"0.82rem\",color:\"var(--gold)\"}}>{playerB.Player}</div>}\n",
    "              </div>\n",
    "            </div>\n",
    "            <button className=\"btn\" onClick={compare} disabled={!playerA||!playerB}>Compare</button>\n",
    "          </div>\n",
    "          {loading ? <Spinner /> : result && result.length === 2 && (\n",
    "            <div className=\"card\">\n",
    "              <table className=\"compare-table\">\n",
    "                <thead>\n",
    "                  <tr>\n",
    "                    <th>Metric</th>\n",
    "                    <th style={{color:\"var(--accent)\"}}>{result[0].Player}</th>\n",
    "                    <th style={{color:\"var(--gold)\"}}>{result[1].Player}</th>\n",
    "                  </tr>\n",
    "                </thead>\n",
    "                <tbody>\n",
    "                  {METRICS.map(({key,label,fmt:f}) => {\n",
    "                    const va = getVal(result[0], key);\n",
    "                    const vb = getVal(result[1], key);\n",
    "                    const na = parseFloat(va), nb = parseFloat(vb);\n",
    "                    const aWins = !isNaN(na) && !isNaN(nb) && na > nb;\n",
    "                    const bWins = !isNaN(na) && !isNaN(nb) && nb > na;\n",
    "                    return (\n",
    "                      <tr key={key}>\n",
    "                        <td>{label}</td>\n",
    "                        <td className={aWins?\"better\":\"\"}>{f(va)}</td>\n",
    "                        <td className={bWins?\"better\":\"\"}>{f(vb)}</td>\n",
    "                      </tr>\n",
    "                    );\n",
    "                  })}\n",
    "                </tbody>\n",
    "              </table>\n",
    "            </div>\n",
    "          )}\n",
    "        </div>\n",
    "      );\n",
    "    };\n",
    "\n",
    "    // ── APP SHELL ────────────────────────────────────────────────────\n",
    "    const App = () => {\n",
    "      const [page, setPage] = useState(\"home\");\n",
    "      const pages = [\n",
    "        {id:\"home\",    label:\"Home\"},\n",
    "        {id:\"recommend\",label:\"Scout\"},\n",
    "        {id:\"similar\", label:\"Similar\"},\n",
    "        {id:\"gems\",    label:\"Gems 💎\"},\n",
    "        {id:\"compare\", label:\"Compare\"},\n",
    "      ];\n",
    "      return (\n",
    "        <div className=\"app\">\n",
    "          <nav className=\"nav\">\n",
    "            <div className=\"nav-logo\" onClick={()=>setPage(\"home\")}>⚽ ScoutIQ</div>\n",
    "            <div className=\"nav-links\">\n",
    "              {pages.map(({id,label}) => (\n",
    "                <button key={id} className={`nav-btn${page===id?\" active\":\"\"}`} onClick={()=>setPage(id)}>{label}</button>\n",
    "              ))}\n",
    "            </div>\n",
    "          </nav>\n",
    "          <main className=\"main\">\n",
    "            {page===\"home\"      && <HomePage setPage={setPage} />}\n",
    "            {page===\"recommend\" && <RecommendPage />}\n",
    "            {page===\"similar\"   && <SimilarPage />}\n",
    "            {page===\"gems\"      && <GemsPage />}\n",
    "            {page===\"compare\"   && <ComparePage />}\n",
    "          </main>\n",
    "        </div>\n",
    "      );\n",
    "    };\n",
    "\n",
    "    ReactDOM.render(<App />, document.getElementById(\"root\"));\n",
    "  </script>\n",
    "</body>\n",
    "</html>'''\n",
    "\n",
    "with open(FRONTEND / 'index.html', 'w', encoding='utf-8') as f:\n",
    "    f.write(html)\n",
    "size = (FRONTEND / 'index.html').stat().st_size\n",
    "print(f'✅ index.html written — {size:,} bytes')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print('='*60)\n",
    "print('  NB09 COMPLETE — REACT FRONTEND')\n",
    "print('='*60)\n",
    "print(f'  File: {FRONTEND / \"index.html\"}')\n",
    "print()\n",
    "print('  Pages:')\n",
    "print('    🏠 Home       — hero + KPIs + quick nav')\n",
    "print('    🎯 Scout      — filter by position/style/budget/age')\n",
    "print('    🔄 Similar    — KNN player lookup')\n",
    "print('    💎 Gems       — hidden talent finder')\n",
    "print('    ⚖️  Compare    — side-by-side stats')\n",
    "print()\n",
    "print('  TO OPEN:')\n",
    "print(f'  1. Make sure API is running: uvicorn src.api.main:app --port 8000')\n",
    "print(f'  2. Open: {FRONTEND / \"index.html\"}')\n",
    "print('     (just double-click or drag into Chrome)')\n",
    "print()\n",
    "print('  → PROJECT COMPLETE: ScoutIQ is ready!')\n",
    "print('='*60)"
   ]
  }
 ],
 "metadata": {\n  "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},\n  "language_info": {"name": "python", "version": "3.12.0"}\n },\n "nbformat": 4,\n "nbformat_minor": 4\n}'''

with open(FRONTEND / 'index.html', 'w', encoding='utf-8') as f:
    f.write(html)
size = (FRONTEND / 'index.html').stat().st_size
print(f'✅ index.html written — {size:,} bytes')

In [ ]:
print('='*60)
print('  NB09 COMPLETE — REACT FRONTEND')
print('='*60)
print(f'  File: {FRONTEND / "index.html"}')
print()
print('  Pages:')
print('    🏠 Home       — hero + KPIs + quick nav')
print('    🎯 Scout      — filter by position/style/budget/age')
print('    🔄 Similar    — KNN player lookup')
print('    💎 Gems       — hidden talent finder')
print('    ⚖️  Compare    — side-by-side stats')
print()
print('  TO OPEN:')
print('  1. Make sure API is running: uvicorn src.api.main:app --port 8000')
print('  2. Open src/frontend/index.html in Chrome')
print()
print('  → PROJECT COMPLETE: ScoutIQ is ready!')
print('='*60)